# 🔗 LLM Orchestration — Hands-On Lab## Build chains, graphs, and pipelines. Compare frameworks side by side.**What this notebook does:**- LCEL chains: basic, parallel, fallback, streaming- LangGraph: state machine with conditional routing- LlamaIndex: complete RAG in 10 lines- Raw API vs LCEL vs wrapper: same task, 3 approaches- Framework decision tool: which one for YOUR use case?**Setup:**```pip install openai langchain langchain-openai langgraph llama-index llama-index-llms-openai llama-index-embeddings-openai```**Cost:** ~$0.30 to run everything

In [ ]:
# ============================================================
# SETUP — Run this first!
# ============================================================

# pip install openai langchain langchain-openai langgraph
from openai import OpenAI
import time, json, asyncio

client = OpenAI()

def ask(prompt, model="gpt-4o-mini", temperature=0):
    r = client.chat.completions.create(
        model=model, temperature=temperature,
        messages=[{"role":"user","content":prompt}])
    return r.choices[0].message.content.strip(), r.usage.total_tokens

answer, _ = ask("Say 'Orchestration ready' in 2 words.")
print(f"Setup complete! {answer}")

---# 1️⃣ LCEL Chains (LangChain Expression Language)**Analogy:** Unix command-line pipes. `cat file | grep error | sort | head`. Each command does one thing. The pipe connects them. Same idea: `prompt | llm | parser`.**What we build:** Basic chain, parallel chain (2 tasks at once), fallback chain (backup plan), and streaming (word by word).

In [ ]:
# ============================================================
# LCEL: Snap components together with the pipe | operator
# prompt | llm | parser = a chain that formats, generates, and extracts
# ============================================================

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
parser = StrOutputParser()

# ---------- BASIC CHAIN (assembly line) ----------

print("1. BASIC CHAIN (prompt → LLM → parse text)")
print("─" * 50)

# Three components snapped together with |
chain = (
    ChatPromptTemplate.from_template("Give me one interesting fact about {topic}")
    | llm
    | parser
)

# .invoke() runs the chain
result = chain.invoke({"topic": "octopuses"})
print(f"   Result: {result[:80]}...")

# The SAME chain automatically supports batch!
print("\n   Batch mode (3 topics at once):")
results = chain.batch([
    {"topic": "Mars"},
    {"topic": "honey"},
    {"topic": "octopuses"},
])
for r in results:
    print(f"   • {r[:60]}...")


# ---------- PARALLEL CHAIN (two tasks at once) ----------

print("\n\n2. PARALLEL CHAIN (two tasks simultaneously)")
print("─" * 50)

summary_chain = ChatPromptTemplate.from_template("Summarize in 1 sentence: {text}") | llm | parser
keyword_chain = ChatPromptTemplate.from_template("Extract 3 keywords from: {text}") | llm | parser

# RunnableParallel runs both at the same time
parallel = RunnableParallel(
    summary=summary_chain,
    keywords=keyword_chain,
)

text = "Machine learning enables computers to learn from data and make predictions without explicit programming."

start = time.time()
result = parallel.invoke({"text": text})
elapsed = time.time() - start

print(f"   Summary:  {result['summary'][:60]}...")
print(f"   Keywords: {result['keywords'][:60]}...")
print(f"   Time: {elapsed:.1f}s (both ran simultaneously!)")


# ---------- FALLBACK CHAIN (backup plan) ----------

print("\n\n3. FALLBACK CHAIN (Plan A → Plan B)")
print("─" * 50)

# Primary: fast model
plan_a = ChatPromptTemplate.from_template("{q}") | ChatOpenAI(model="gpt-4o-mini") | parser

# Backup: different model (or same model as demo)
plan_b = ChatPromptTemplate.from_template("Please answer: {q}") | ChatOpenAI(model="gpt-4o-mini") | parser

# If plan_a fails, automatically try plan_b
reliable = plan_a.with_fallbacks([plan_b])

result = reliable.invoke({"q": "What is 2+2?"})
print(f"   Result: {result}")
print(f"   (If primary fails, backup kicks in automatically)")


# ---------- STREAMING (word by word) ----------

print("\n\n4. STREAMING (words appear one by one)")
print("─" * 50)

stream_chain = ChatPromptTemplate.from_template("Tell me a fun fact about {topic} in 2 sentences.") | llm | parser

print("   ", end="")
for chunk in stream_chain.stream({"topic": "dolphins"}):
    print(chunk, end="", flush=True)
    time.sleep(0.02)  # Small delay so you can see it streaming
print()

print("\n\n💡 Every LCEL chain automatically gets: .invoke(), .batch(), .stream()")
print(f"💡 RunnableParallel: same cost but 2-3x faster for independent tasks.")
print(f"💡 .with_fallbacks(): production essential. Never depend on one provider.")

---# 2️⃣ LangGraph (Stateful Agents with Routing)**Analogy:** LCEL is a straight highway. LangGraph is a full road network with intersections, roundabouts, and rest stops. You need it when you have routing ("if billing → go left"), loops ("try again"), or state ("remember what happened").**What we build:** Customer support system: classify → route → handle → quality check → respond.

In [ ]:
# ============================================================
# LANGGRAPH: A flowchart you can run
# Nodes = functions. Edges = connections. Conditional edges = routing.
# ============================================================

from typing import TypedDict, Literal
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# ---------- STATE (shared clipboard) ----------

class TicketState(TypedDict):
    query: str
    category: str
    priority: str
    context: str
    response: str
    quality_score: int

# ---------- NODES (boxes in the flowchart) ----------

def classify(state: TicketState) -> TicketState:
    """Node 1: Classify the ticket."""
    result = llm.invoke(
        f"Classify as billing/technical/general AND priority high/medium/low.\n"
        f"Query: {state['query']}\n"
        f"Format: category|priority"
    ).content.strip().lower()
    
    parts = result.split("|")
    state["category"] = parts[0].strip() if len(parts) > 0 else "general"
    state["priority"] = parts[1].strip() if len(parts) > 1 else "medium"
    print(f"    Classified: {state['category']} | {state['priority']}")
    return state

def handle_billing(state: TicketState) -> TicketState:
    state["context"] = "Pro:$49/mo, Enterprise:$199/mo, Annual:20% off, Student:50% off"
    state["response"] = llm.invoke(
        f"Billing specialist. Context: {state['context']}\nQuery: {state['query']}\nBe concise."
    ).content.strip()
    print(f"    💰 Billing handled")
    return state

def handle_technical(state: TicketState) -> TicketState:
    state["context"] = "Common fixes: restart app, clear cache, update version. Docs: docs.example.com"
    state["response"] = llm.invoke(
        f"Tech support. Context: {state['context']}\nQuery: {state['query']}\nBe concise."
    ).content.strip()
    print(f"    🔧 Technical handled")
    return state

def handle_general(state: TicketState) -> TicketState:
    state["response"] = llm.invoke(
        f"Friendly support agent. Query: {state['query']}\nBe concise."
    ).content.strip()
    print(f"    💬 General handled")
    return state

def quality_check(state: TicketState) -> TicketState:
    """Node 3: Check response quality before sending."""
    score_text = llm.invoke(
        f"Rate this response 1-10 on helpfulness.\n"
        f"Question: {state['query']}\nResponse: {state['response']}\nNumber only."
    ).content.strip()
    try:
        state["quality_score"] = int(score_text.split()[0])
    except:
        state["quality_score"] = 7
    print(f"    Quality: {state['quality_score']}/10")
    return state

# ---------- ROUTING (the fork in the road) ----------

def route_category(state: TicketState) -> Literal["billing", "technical", "general"]:
    cat = state.get("category", "general")
    if "bill" in cat: return "billing"
    if "tech" in cat: return "technical"
    return "general"

# ---------- BUILD THE GRAPH ----------

graph = StateGraph(TicketState)

# Add nodes
graph.add_node("classify", classify)
graph.add_node("billing", handle_billing)
graph.add_node("technical", handle_technical)
graph.add_node("general", handle_general)
graph.add_node("quality_check", quality_check)

# Connect them
graph.set_entry_point("classify")
graph.add_conditional_edges("classify", route_category)

# All handlers → quality check → END
for handler in ["billing", "technical", "general"]:
    graph.add_edge(handler, "quality_check")
graph.add_edge("quality_check", END)

# Compile
app = graph.compile()

# ---------- TEST ----------

print("LANGGRAPH: Customer Support Pipeline")
print("=" * 60)

queries = [
    "How much does the Enterprise plan cost with annual billing?",
    "API returns 500 error when I try to upload files",
    "Just wanted to say your product is awesome!",
]

for q in queries:
    print(f"\n  User: {q}")
    result = app.invoke({
        "query": q, "category": "", "priority": "",
        "context": "", "response": "", "quality_score": 0
    })
    print(f"  Response: {result['response'][:80]}...")
    print(f"  Quality: {result['quality_score']}/10 | Category: {result['category']}")

print("\n💡 classify → route to right handler → quality check → respond.")
print("💡 Add checkpointing to survive server restarts.")
print("💡 Add interrupt_before=['quality_check'] for human review before sending.")

---# 3️⃣ LlamaIndex (Complete RAG in 10 Lines)**Analogy:** LangChain is a general-purpose toolbox. LlamaIndex is a specialized RAG workshop. It was built specifically for connecting LLMs to YOUR data. What takes 100 lines in LangChain takes 10 in LlamaIndex.**What we build:** A working RAG system with sources and relevance scores. Minimal code.

In [ ]:
# ============================================================
# LLAMAINDEX: The RAG-specialized framework
# Same RAG pipeline as Lab 2, but in ~10 lines
# ============================================================

try:
    from llama_index.core import VectorStoreIndex, Document, Settings
    from llama_index.llms.openai import OpenAI as LlamaOpenAI
    from llama_index.embeddings.openai import OpenAIEmbedding
    
    # Configure (2 lines)
    Settings.llm = LlamaOpenAI(model="gpt-4o-mini")
    Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")
    
    # Create documents (your knowledge base)
    documents = [
        Document(text="Refund policy: 30 days for standard items. Electronics: 15 days. Must be unused with receipt."),
        Document(text="Pricing: Pro plan $49/month. Enterprise $199/month. Annual billing: 20% discount."),
        Document(text="Warranty: 1 year for manufacturing defects. Physical damage NOT covered. Extended warranty $99/year."),
        Document(text="Shipping: Free over $50. Standard 5-7 days. Express 2-day for $15. International to 30 countries."),
    ]
    
    # Build index + query engine (2 lines — ALL the RAG plumbing!)
    index = VectorStoreIndex.from_documents(documents)
    engine = index.as_query_engine(similarity_top_k=2)
    
    # Ask questions
    print("LLAMAINDEX RAG (10 lines of code!)")
    print("=" * 60)
    
    questions = [
        "Can I return electronics after 20 days?",
        "How much is Pro plan for a year?",
        "Is physical damage covered?",
        "What is your return policy for Mars?",  # Not in docs
    ]
    
    for q in questions:
        response = engine.query(q)
        print(f"\n  Q: {q}")
        print(f"  A: {response.response[:80]}...")
        
        # Show which documents were used
        for node in response.source_nodes:
            print(f"     Source [{node.score:.3f}]: {node.text[:50]}...")
    
    print("\n💡 Complete RAG in ~10 lines. LlamaIndex handles chunking, embedding, retrieval, synthesis.")
    print("💡 .source_nodes shows which documents were used (with relevance scores).")

except ImportError:
    print("LlamaIndex not installed. Run: pip install llama-index llama-index-llms-openai llama-index-embeddings-openai")
    print("\nHere is what the code looks like (10 lines for a full RAG system):")
    print("""
    Settings.llm = OpenAI(model="gpt-4o-mini")
    Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")
    documents = [Document(text="your docs here")]
    index = VectorStoreIndex.from_documents(documents)
    engine = index.as_query_engine(similarity_top_k=2)
    response = engine.query("your question")
    """)

---# 4️⃣ Raw API vs LCEL vs Wrapper (Same Task, 3 Ways)**Analogy:** Going to the grocery store: walk (raw API), drive a car (LCEL), or take a delivery truck (framework). Each has its place. Walking is fine for a quick trip. The truck is overkill for milk.**What we compare:** The EXACT same classification task done 3 ways. See the code complexity and when each makes sense.

In [ ]:
# ============================================================
# SAME TASK, 3 APPROACHES
# Task: Classify customer query as billing/technical/general
# ============================================================

query = "How much does the Enterprise plan cost per year?"

print("SAME TASK — 3 APPROACHES")
print("=" * 60)

# ---------- APPROACH 1: RAW API (the walk) ----------

print("\n1. RAW API (simplest — 5 lines)")
print("─" * 40)

start = time.time()
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role":"user","content":f"Classify as billing/technical/general. One word.\n{query}"}],
    temperature=0,
)
raw_result = response.choices[0].message.content.strip()
raw_time = time.time() - start
raw_tokens = response.usage.total_tokens

print(f"   Result: {raw_result}")
print(f"   Time: {raw_time:.2f}s | Tokens: {raw_tokens}")
print(f"   Lines of code: 5")
print(f"   Pros: full control, no dependencies, easy to debug")
print(f"   Cons: no streaming/batch/fallback for free")


# ---------- APPROACH 2: LCEL (the car) ----------

print("\n2. LCEL CHAIN (adds streaming, batch, fallback)")
print("─" * 40)

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

chain = (
    ChatPromptTemplate.from_template("Classify as billing/technical/general. One word.\n{query}")
    | ChatOpenAI(model="gpt-4o-mini", temperature=0)
    | StrOutputParser()
)

start = time.time()
lcel_result = chain.invoke({"query": query})
lcel_time = time.time() - start

print(f"   Result: {lcel_result}")
print(f"   Time: {lcel_time:.2f}s")
print(f"   Lines of code: 5")
print(f"   Pros: .batch(), .stream(), .with_fallbacks() for FREE")
print(f"   Cons: extra dependency, abstraction layer to debug through")


# ---------- APPROACH 3: FRAMEWORK-AGNOSTIC WRAPPER ----------

print("\n3. YOUR OWN WRAPPER (lock-in prevention)")
print("─" * 40)

class LLMService:
    """Your interface. Swap providers by changing ONE class."""
    
    def __init__(self, provider="openai"):
        self.provider = provider
        self.client = OpenAI()
    
    def classify(self, query: str) -> str:
        r = self.client.chat.completions.create(
            model="gpt-4o-mini", temperature=0,
            messages=[{"role":"user","content":f"Classify billing/technical/general. One word.\n{query}"}]
        )
        return r.choices[0].message.content.strip()

service = LLMService()
start = time.time()
wrapper_result = service.classify(query)
wrapper_time = time.time() - start

print(f"   Result: {wrapper_result}")
print(f"   Time: {wrapper_time:.2f}s")
print(f"   Lines of code: 10")
print(f"   Pros: switch provider = change 1 class. No lock-in.")
print(f"   Cons: you build streaming/batch yourself")


# ---------- COMPARISON TABLE ----------

print(f"\n\nCOMPARISON:")
print(f"{'Approach':<30} {'Result':<12} {'Time':>6} {'Dependencies':>15}")
print("─" * 65)
print(f"{'Raw API (walk)':<30} {raw_result:<12} {raw_time:>5.2f}s {'openai':>15}")
print(f"{'LCEL (car)':<30} {lcel_result:<12} {lcel_time:>5.2f}s {'langchain':>15}")
print(f"{'Wrapper (your interface)':<30} {wrapper_result:<12} {wrapper_time:>5.2f}s {'openai':>15}")

print(f"\n💡 All 3 produce the SAME result. Choose based on what you NEED:")
print(f"   Simple single call? → Raw API")
print(f"   Need streaming + batch + fallback? → LCEL")
print(f"   Need provider flexibility? → Your own wrapper")
print(f"   Building agents? → LangGraph (next level up from LCEL)")

---# 5️⃣ Framework Decision Helper**The biggest mistake:** Picking a framework BEFORE understanding the problem. Sometimes a 3-line API call is all you need. Adding a framework to a simple problem is like using a forklift to move a coffee cup.**This tool:** Answer 4 questions about your project → get a framework recommendation.

In [ ]:
# ============================================================
# FRAMEWORK DECISION HELPER
# Answer questions about your project → get a recommendation
# ============================================================

def framework_decision(
    needs_routing: bool,
    needs_memory: bool,
    needs_human_approval: bool,
    needs_rag: bool,
    needs_multi_agent: bool,
    query_volume: str,  # "low" (<1K/day), "medium" (1K-100K), "high" (>100K)
):
    """Recommend the right framework based on your actual needs."""
    
    score = {
        "Raw API": 0,
        "LCEL (LangChain)": 0,
        "LangGraph": 0,
        "LlamaIndex": 0,
    }
    reasons = {k: [] for k in score}
    
    # Simple cases favor raw API
    if not any([needs_routing, needs_memory, needs_human_approval, needs_multi_agent]):
        score["Raw API"] += 5
        reasons["Raw API"].append("Simple use case — no routing, memory, or approval needed")
    
    # Routing, loops, state → LangGraph
    if needs_routing:
        score["LangGraph"] += 3
        reasons["LangGraph"].append("Conditional routing between handlers")
    if needs_memory:
        score["LangGraph"] += 2
        reasons["LangGraph"].append("State persistence across interactions")
    if needs_human_approval:
        score["LangGraph"] += 3
        reasons["LangGraph"].append("Human-in-the-loop approval gates")
    if needs_multi_agent:
        score["LangGraph"] += 2
        reasons["LangGraph"].append("Multi-agent orchestration")
    
    # RAG → LlamaIndex
    if needs_rag:
        score["LlamaIndex"] += 4
        reasons["LlamaIndex"].append("RAG is the primary use case")
        score["LCEL (LangChain)"] += 2
        reasons["LCEL (LangChain)"].append("Can do RAG with more manual wiring")
    
    # LCEL for middle ground
    if not needs_routing and not needs_human_approval:
        score["LCEL (LangChain)"] += 2
        reasons["LCEL (LangChain)"].append("Streaming + batch + fallback without full graph")
    
    # Volume considerations
    if query_volume == "low":
        score["Raw API"] += 2
        reasons["Raw API"].append("Low volume — simplicity wins")
    
    # Sort by score
    ranked = sorted(score.items(), key=lambda x: x[1], reverse=True)
    
    return ranked, reasons


# ---------- TEST 3 SCENARIOS ----------

print("FRAMEWORK DECISION HELPER")
print("=" * 60)

scenarios = [
    {
        "name": "Simple Chatbot",
        "desc": "Basic Q&A bot, no routing, no approval, low volume",
        "args": {"needs_routing":False, "needs_memory":False, "needs_human_approval":False,
                 "needs_rag":False, "needs_multi_agent":False, "query_volume":"low"}
    },
    {
        "name": "Customer Support System", 
        "desc": "Route by department, human approval for refunds, RAG for policies",
        "args": {"needs_routing":True, "needs_memory":True, "needs_human_approval":True,
                 "needs_rag":True, "needs_multi_agent":False, "query_volume":"medium"}
    },
    {
        "name": "Document Q&A Tool",
        "desc": "Search company docs, answer questions, no routing needed",
        "args": {"needs_routing":False, "needs_memory":False, "needs_human_approval":False,
                 "needs_rag":True, "needs_multi_agent":False, "query_volume":"medium"}
    },
]

for scenario in scenarios:
    print(f"\n  Scenario: {scenario['name']}")
    print(f"  {scenario['desc']}")
    print(f"  {'─' * 50}")
    
    ranked, reasons = framework_decision(**scenario["args"])
    
    for i, (framework, score) in enumerate(ranked):
        medal = ["🥇","🥈","🥉","  "][i]
        print(f"  {medal} {framework:<25} (score: {score})")
        for reason in reasons[framework]:
            print(f"       → {reason}")

print(f"\n\n  DECISION CHEAT SHEET:")
print(f"  ─" * 30)
print(f"  Simple Q&A, single call       → Raw API")
print(f"  Need streaming + batch         → LCEL")
print(f"  Routing + state + approval     → LangGraph")
print(f"  RAG-focused app                → LlamaIndex")
print(f"  RAG + routing + agents         → LangGraph + LlamaIndex together")
print(f"\n💡 Start simple. Add complexity only when you feel the pain it solves.")
print(f"💡 Wrap framework calls in YOUR interface. Switch frameworks ≠ rewrite app.")

---# 6️⃣ Anti-Patterns (Mistakes to Avoid)**The 5 most common mistakes** teams make when choosing and using orchestration frameworks. Each one shown with code: the wrong way vs the right way.

In [ ]:
# ============================================================
# 5 ANTI-PATTERNS: Common mistakes and how to avoid them
# ============================================================

print("ORCHESTRATION ANTI-PATTERNS")
print("=" * 60)


# ANTI-PATTERN 1: Framework for a simple task
print("\n❌ ANTI-PATTERN 1: Using a framework for a 3-line task")
print("─" * 50)

print("  WRONG (15 lines with LangChain):")
print("""    from langchain_openai import ChatOpenAI
    from langchain_core.prompts import ChatPromptTemplate
    from langchain_core.output_parsers import StrOutputParser
    chain = ChatPromptTemplate.from_template("{q}") | ChatOpenAI() | StrOutputParser()
    result = chain.invoke({"q": "What is 2+2?"})
""")

print("  RIGHT (3 lines with raw API):")
print("""    r = client.chat.completions.create(model="gpt-4o-mini", messages=[{"role":"user","content":"What is 2+2?"}])
    result = r.choices[0].message.content
""")
print("  Rule: if raw API is simpler, use raw API.\n")


# ANTI-PATTERN 2: Not pinning versions
print("\n❌ ANTI-PATTERN 2: Not pinning framework versions")
print("─" * 50)
print("  WRONG: pip install langchain")
print("  RIGHT: pip install langchain==0.3.7")
print("  Why: LangChain changes APIs frequently. Unpinned = surprise breakage.\n")


# ANTI-PATTERN 3: Vendor lock-in
print("\n❌ ANTI-PATTERN 3: Framework lock-in (200 direct calls)")
print("─" * 50)

print("  WRONG (locked to LangChain):")
print("    # 200 files all import from langchain directly")
print("    # Switching = rewrite everything\n")

print("  RIGHT (your interface wraps the framework):")
print("""    class LLMService:
        def classify(self, text):
            # LangChain internally (or raw API, or LlamaIndex)
            # Change HERE, not in 200 files
            return chain.invoke({"text": text})
""")


# ANTI-PATTERN 4: Multiple frameworks
print("\n❌ ANTI-PATTERN 4: Mixing 3 frameworks in one project")
print("─" * 50)
print("  WRONG: LangChain for chains + LlamaIndex for RAG + Haystack for search")
print("  RIGHT: Pick ONE primary framework. LangGraph for agents + LlamaIndex for RAG is OK.")
print("  Rule: max 2 frameworks, with clear boundaries.\n")


# ANTI-PATTERN 5: Not testing without the framework
print("\n❌ ANTI-PATTERN 5: Debugging through 5 abstraction layers")
print("─" * 50)
print("  Symptom: 'I got an error but I don't know which layer caused it'")
print("  Fix: when debugging, REMOVE the framework. Make the raw API call.")
print("  If raw API works but framework doesn't → framework config issue.")
print("  If raw API fails too → your prompt/data is the problem.\n")


# SUMMARY
print("\n✅ RULES TO LIVE BY:")
print("  1. Start with raw API. Add framework when you feel the pain.")
print("  2. Pin versions. Test before upgrading.")
print("  3. Wrap framework calls in YOUR interface.")
print("  4. Max 2 frameworks per project.")
print("  5. When debugging, test with raw API first.")

---# 🏁 Summary — Orchestration Decision Map| Need | Framework | Why ||------|-----------|-----|| Simple single call | Raw API | No overhead, full control || Streaming + batch + fallback | LCEL (LangChain) | Free with pipe syntax || Routing + loops + state | LangGraph | Production agent standard || RAG-focused app | LlamaIndex | 10x less code for RAG || Microsoft/Azure stack | Semantic Kernel | Native Azure integration || Type-safe NLP pipelines | Haystack | Build-time validation |**The ladder:** Raw API → LCEL → LangGraph. Stop at the simplest that works.**Non-negotiable:** Wrap framework calls in YOUR interface. This makes switching frameworks a 1-file change, not a 200-file rewrite.